### 1. Bloque del PVC.

In [29]:
import pyodbc
import pandas as pd

print("1. Conectando a MICROLOGS y extrayendo datos de Agil_Logs...")

conexion_logs = (
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=FABOGSQL00\MICROLOGS,53981;'
    r'DATABASE=Agil_Logs;'
    r'Trusted_Connection=yes;'
)

query_pvc = """
WITH TrazaFiltrada AS ( 
    SELECT 
        Solicitud, Fecha, CAST(Fecha AS DATE) AS FechaDato, Producto, Procedimiento, Parametros, Resultado 
    FROM [Traza].[trazaPVC] 
    WHERE Fecha >= '2026-01-05'  
      AND (
          Procedimiento = 'AprobarProducto'
          OR Procedimiento LIKE '%MontoMaximoDefinitivoSinCP%'
          OR Procedimiento LIKE '%MontoMaximoDefinitivoConCP%'
          OR Procedimiento = 'EndeudamientoOtorgamiento DisponibleProducto'
      )
),
DatosAgrupados AS (
    SELECT 
        Solicitud, FechaDato AS Fecha, MAX(Fecha) AS MaxFechaExacta, 
        MAX(Producto) AS Producto, -- Se extrae el nombre del producto de la base de datos
        MAX(CASE WHEN Producto = 'Vehiculo' THEN 1 ELSE 0 END) AS Flag_Vehiculo,
        MAX(CASE WHEN Producto LIKE '%Tarjeta%' THEN 1 ELSE 0 END) AS Flag_Tarjeta,
        MAX(CASE WHEN Procedimiento LIKE '%MontoMaximoDefinitivoSinCP%' THEN Resultado END) AS DefinitivoSinCP,
        MAX(CASE WHEN Procedimiento LIKE '%MontoMaximoDefinitivoConCP%' THEN Resultado END) AS DefinitivoConCp,
        MAX(CASE WHEN Procedimiento = 'AprobarProducto' THEN Resultado END) AS Estado_Aprobacion,
        MAX(CASE WHEN Procedimiento = 'EndeudamientoOtorgamiento DisponibleProducto' THEN Resultado END) AS Capacidad_Pago,
        
        -- 1. Oferta Vehículo
        MAX(CASE WHEN Procedimiento = 'AprobarProducto' AND CHARINDEX('OfertaValida=', Parametros) > 0 
            THEN TRY_CAST(REPLACE(SUBSTRING(Parametros, CHARINDEX('OfertaValida=', Parametros) + 13, CHARINDEX(';', Parametros + ';', CHARINDEX('OfertaValida=', Parametros)) - (CHARINDEX('OfertaValida=', Parametros) + 13)), ',', '.') AS FLOAT) 
        END) AS OfertaValida_Vehiculo,

        -- 2. Oferta Tarjeta
        MIN(CASE WHEN Procedimiento = 'AprobarProducto' AND CHARINDEX('OfertaValida=', Parametros) > 0 
            THEN TRY_CAST(REPLACE(SUBSTRING(Parametros, CHARINDEX('OfertaValida=', Parametros) + 13, CHARINDEX(';', Parametros + ';', CHARINDEX('OfertaValida=', Parametros)) - (CHARINDEX('OfertaValida=', Parametros) + 13)), ',', '.') AS FLOAT) 
        END) AS OfertaValida_Tarjeta_Bruta

    FROM TrazaFiltrada 
    GROUP BY Solicitud, FechaDato 
),
UltimaTraza AS (
    SELECT 
        MaxFechaExacta AS Fecha_Ultima_Traza, 
        Solicitud, Producto, Flag_Vehiculo, Flag_Tarjeta, DefinitivoSinCP, DefinitivoConCp, 
        OfertaValida_Vehiculo, OfertaValida_Tarjeta_Bruta, Capacidad_Pago, Estado_Aprobacion,
        ROW_NUMBER() OVER(PARTITION BY Solicitud ORDER BY MaxFechaExacta DESC) as rn
    FROM DatosAgrupados
)
SELECT 
    Fecha_Ultima_Traza AS Fecha, 
    Solicitud, 
    Producto, -- Se imprime la columna en el resultado final
    CASE 
        WHEN Flag_Vehiculo = 1 AND Flag_Tarjeta = 1 THEN 'Vehículo + Tarjeta (Cruzada)'
        WHEN Flag_Vehiculo = 1 AND Flag_Tarjeta = 0 THEN 'Solo Vehículo'
        WHEN Flag_Vehiculo = 0 AND Flag_Tarjeta = 1 THEN 'Solo Tarjeta'
        ELSE 'Otro' 
    END AS Tipo_Solicitud,
    DefinitivoSinCP, DefinitivoConCp, 
    
    OfertaValida_Vehiculo,
    CASE 
        WHEN OfertaValida_Vehiculo <> OfertaValida_Tarjeta_Bruta THEN OfertaValida_Tarjeta_Bruta 
        ELSE NULL 
    END AS OfertaValida_Tarjeta,
    
    Capacidad_Pago, Estado_Aprobacion
FROM UltimaTraza 
WHERE rn = 1;
"""

conn_logs = pyodbc.connect(conexion_logs)
df_pvc = pd.read_sql(query_pvc, conn_logs)
conn_logs.close()

print(f"Extracción completada. Filas obtenidas: {len(df_pvc)}")
df_pvc.head()

1. Conectando a MICROLOGS y extrayendo datos de Agil_Logs...


C:\Users\osccam\AppData\Local\Temp\ipykernel_19244\224121984.py:82: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_pvc = pd.read_sql(query_pvc, conn_logs)


Extracción completada. Filas obtenidas: 274058


,Fecha,Solicitud,Producto,Tipo_Solicitud,DefinitivoSinCP,DefinitivoConCp,OfertaValida_Vehiculo,OfertaValida_Tarjeta,Capacidad_Pago,Estado_Aprobacion
0,2026-06-25 15:29:55.163,5545975,Vehiculo,Vehículo + Tarjeta (Cruzada),1000000,8400000,1.232000e+08,8400000.0,"3743626,6",Aprobado
1,2026-01-27 10:49:12.680,6177875,Vehiculo,Vehículo + Tarjeta (Cruzada),"4805252,64",6000000,8.800000e+07,6000000.0,"3559446,4",Aprobado
2,2026-01-13 13:49:22.190,6188131,Vehiculo,Vehículo + Tarjeta (Cruzada),9763740,5802500,8.000000e+07,0.0,6328350,Denegado
3,2026-01-07 12:04:57.197,6212828,Vehiculo,Vehículo + Tarjeta (Cruzada),3931200,17998000,8.956958e+07,12740000.0,2548000,Aprobado
4,2026-01-06 11:58:15.470,6221346,Vehiculo,Vehículo + Tarjeta (Cruzada),"2926439,82",0,2.878003e+07,0.0,"1354833,25",Denegado


### 2. Bloque BeClever

In [30]:
import pyodbc
import pandas as pd

print("2. Conectando a Fabogcubox y extrayendo datos de Finandina_cartera (Última Traza)...")

conexion_cartera = (
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=Fabogcubox;'
    r'DATABASE=Finandina_cartera;'
    r'Trusted_Connection=yes;'
)

query_riesgo = """
WITH RiesgoHistorico AS (
    SELECT 
        [NoSolicitud], 
        [o_RiesgoSolicitud] AS Letra_Riesgo,
        [FechaRegistro], 
        [FechaSolicitud],
        -- Ordenamos por FechaRegistro para traer la última actualización de la solicitud
        ROW_NUMBER() OVER(PARTITION BY [NoSolicitud] ORDER BY [FechaRegistro] DESC) as rn
    FROM dbo.DB_BeCleverRegistro
    
    -- NUEVO FILTRO: El corte del 5 de enero ahora aplica sobre la Fecha de Solicitud
    WHERE [FechaSolicitud] >= '2026-01-05'
)
SELECT 
    [NoSolicitud], 
    Letra_Riesgo,
    [FechaRegistro] AS Fecha_Registro_BeClever,
    [FechaSolicitud] AS Fecha_Solicitud_BeClever
FROM RiesgoHistorico
WHERE rn = 1;
"""

conn_cartera = pyodbc.connect(conexion_cartera)
df_riesgo = pd.read_sql(query_riesgo, conn_cartera)
conn_cartera.close()

print(f"Extracción completada. Filas únicas obtenidas: {len(df_riesgo)}")
df_riesgo.head()

2. Conectando a Fabogcubox y extrayendo datos de Finandina_cartera (Última Traza)...


C:\Users\osccam\AppData\Local\Temp\ipykernel_19244\4207431691.py:37: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_riesgo = pd.read_sql(query_riesgo, conn_cartera)


Extracción completada. Filas únicas obtenidas: 472973


,NoSolicitud,Letra_Riesgo,Fecha_Registro_BeClever,Fecha_Solicitud_BeClever
0,3127311,Z,2026-01-06 14:40:49.310,2026-01-06
1,3130937,Y,2026-01-06 00:16:36.093,2026-01-06
2,3130953,Z,2026-01-05 23:58:35.683,2026-01-05
3,3130992,Y,2026-01-06 00:00:11.227,2026-01-06
4,3131002,A,2026-01-06 00:00:40.237,2026-01-06


### 3. Bloque Montos

In [31]:
print("3. Conectando al Servidor Principal y extrayendo Montos desde Linked Server...")

# Usamos el mismo servidor del Bloque 1, ya que desde allí se invoca el Linked Server
conexion_montos = (
    r'DRIVER={ODBC Driver 17 for SQL Server};'
    r'SERVER=Fabogcubox;'
    r'DATABASE=Finandina_cartera;'
    r'Trusted_Connection=yes;'
)

query_montos = """
WITH Monto AS (
    SELECT *
    FROM (
        SELECT 
            sNumerodeCasoConsulta AS NoSolicitud,
            CAST(b.dfechadesolicitud AS DATE) AS FechaSolicitud,
            sNumeroidentificacion,
            idP_TipoActividadEconomica,
            TA.Nombre,
            X.idM_Solicitud_VFF,
            cMontoSolicitado AS MontoSolicitado,
            cMontoDesembolsado AS MontoFabrica,
            cMontoAprobado AS MontoMotor,
            y.bEsProductoPrincipal AS prd_princpipal,
            CAST(b.sNumeroIdParticipantePpal AS NVARCHAR) AS par_principal
        FROM [AZURE_BIZAGISQL_LINK].bmdeaujxdbz3gu.dbo.[M_Participantes] C
        INNER JOIN [AZURE_BIZAGISQL_LINK].bmdeaujxdbz3gu.dbo.M_Solicitud_VFF X ON C.M_Solicitud_VFF = X.idM_Solicitud_VFF
        INNER JOIN [AZURE_BIZAGISQL_LINK].bmdeaujxdbz3gu.dbo.[DatosGenerales] B ON B.idM_Solicitud_VFF = X.idM_Solicitud_VFF
        INNER JOIN [AZURE_BIZAGISQL_LINK].bmdeaujxdbz3gu.dbo.[M_PersonaNatural_FF] PN ON idM_PersonaNatural_FF = idM_PersonaNatural
        INNER JOIN [AZURE_BIZAGISQL_LINK].bmdeaujxdbz3gu.dbo.[M_ActividadEconomicaFF] AE ON M_ActividadEconomica = idM_ActividadEconomicaFF
        LEFT JOIN [AZURE_BIZAGISQL_LINK].bmdeaujxdbz3gu.dbo.TipoActividadEconomica TA ON idtipoactividadeconomica = idP_TipoActividadEconomica
        LEFT JOIN [AZURE_BIZAGISQL_LINK].bmdeaujxdbz3gu.dbo.[M_Producto_FF] Y ON X.idM_Solicitud_VFF = Y.M_Solicitud_VFF
        
        UNION ALL
        
        SELECT 
            CAST(CAST(nUMEROSOLICITUD AS BIGINT) AS NVARCHAR) AS NoSolicitud,
            CAST(s.fechasolicitud AS DATE) AS FechaSolicitud,
            NumeroIdentificacion,
            AE.TipoActividadEconomica,
            TA.Nombre,
            S.idSolicitud,
            MontoSolicitado,
            MontoAprobado AS MontoFabrica,  
            MontoAprobadoTEMP AS MontoMotor,
            ya.EsProductoPrincipal AS prd_princpipal,
            CAST(s.ParticipantePrincipal AS NVARCHAR) AS par_principal
        FROM [AZURE_BIZAGISQL_LINK].bmdeaujxdbz3gu.dbo.Participante P
        INNER JOIN [AZURE_BIZAGISQL_LINK].bmdeaujxdbz3gu.dbo.Solicitud S ON P.[idParticipante] = S.[ParticipantePrincipal]
        INNER JOIN [AZURE_BIZAGISQL_LINK].bmdeaujxdbz3gu.dbo.[ActividadEconomica] AE ON idActividadEconomica = ActividadEconomica
        LEFT JOIN [AZURE_BIZAGISQL_LINK].bmdeaujxdbz3gu.dbo.TipoActividadEconomica TA ON iDTipoActividadEconomica = AE.TipoActividadEconomica
        LEFT JOIN [AZURE_BIZAGISQL_LINK].bmdeaujxdbz3gu.dbo.[ProductoSolicitado] YA ON S.idSolicitud = YA.Solicitud
    ) A
    -- Filtro de fecha para alinearlo con el resto del ETL
    WHERE FechaSolicitud >= '2025-05-01' AND prd_princpipal = 1
)
SELECT DISTINCT 
    Nosolicitud, 
    MontoSolicitado, 
    MontoFabrica, 
    MontoMotor 
FROM Monto;
"""

conn_montos = pyodbc.connect(conexion_montos)
df_montos = pd.read_sql(query_montos, conn_montos)
conn_montos.close()

print(f"Extracción completada. Filas obtenidas: {len(df_montos)}")
df_montos.head()

3. Conectando al Servidor Principal y extrayendo Montos desde Linked Server...


C:\Users\osccam\AppData\Local\Temp\ipykernel_19244\2632172216.py:67: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_montos = pd.read_sql(query_montos, conn_montos)


Extracción completada. Filas obtenidas: 807486


,Nosolicitud,MontoSolicitado,MontoFabrica,MontoMotor
0,VH6210292,14298000.0,14298000.0,0.0
1,6501318,50400000.0,50400000.0,55400000.0
2,VH6647523,0.0,NaN,0.0
3,5820437,57000000.0,11800000.0,11800000.0
4,VH6164923,51120000.0,51120000.0,0.0


### 4. Cruce.

In [32]:
print("4. Cruzando la información (PVC + Cartera + Montos)...")

# ==========================================
# 1. PRIMER CRUCE: PVC con Riesgo (Cartera)
# ==========================================
df_final = pd.merge(
    df_pvc, 
    df_riesgo, 
    left_on='Solicitud', 
    right_on='NoSolicitud', 
    how='inner' 
)

if 'NoSolicitud' in df_final.columns:
    df_final = df_final.drop(columns=['NoSolicitud'])

# ==========================================
# 2. SEGUNDO CRUCE: Resultado con Montos
# ==========================================
df_final['Solicitud'] = df_final['Solicitud'].astype(str)
df_montos['Nosolicitud'] = df_montos['Nosolicitud'].astype(str)

df_final = pd.merge(
    df_final, 
    df_montos, 
    left_on='Solicitud', 
    right_on='Nosolicitud', 
    how='left'
)

if 'Nosolicitud' in df_final.columns:
    df_final = df_final.drop(columns=['Nosolicitud'])

# ==========================================
# 3. REGLAS DE NEGOCIO Y LIMPIEZA
# ==========================================
# Libranza
df_final.loc[df_final['Producto'].str.contains('Libranza', case=False, na=False), 'Letra_Riesgo'] = None

# Corrección de decimales (SQL separador de miles)
df_final.loc[df_final['OfertaValida_Tarjeta'] > 500000000, 'OfertaValida_Tarjeta'] = df_final['OfertaValida_Tarjeta'] / 1000
df_final.loc[df_final['OfertaValida_Vehiculo'] > 500000000, 'OfertaValida_Vehiculo'] = df_final['OfertaValida_Vehiculo'] / 1000

# ==========================================
# 4. ORDEN FINAL DE COLUMNAS
# ==========================================
columnas = [
    'Fecha', 
    'Fecha_Registro_BeClever',   
    'Fecha_Solicitud_BeClever',  
    'Solicitud',
    'Producto',
    'Letra_Riesgo', 
    'Tipo_Solicitud', 
    'DefinitivoSinCP', 
    'DefinitivoConCp', 
    'OfertaValida_Vehiculo',
    'OfertaValida_Tarjeta',
    'Capacidad_Pago', 
    'MontoSolicitado',       
    'MontoFabrica',          
    'MontoMotor',            
    'Estado_Aprobacion'
]
df_final = df_final[columnas]

print(f"¡Triple cruce finalizado! Filas totales: {len(df_final)}")
df_final.head(10)

4. Cruzando la información (PVC + Cartera + Montos)...
¡Triple cruce finalizado! Filas totales: 259725


,Fecha,Fecha_Registro_BeClever,Fecha_Solicitud_BeClever,Solicitud,Producto,Letra_Riesgo,Tipo_Solicitud,DefinitivoSinCP,DefinitivoConCp,OfertaValida_Vehiculo,OfertaValida_Tarjeta,Capacidad_Pago,MontoSolicitado,MontoFabrica,MontoMotor,Estado_Aprobacion
0,2026-01-07 12:04:57.197,2026-01-07 12:04:52.153,2026-01-07 12:04:51.433,6212828,Vehiculo,A,Vehículo + Tarjeta (Cruzada),3931200,17998000,8.956958e+07,1.274000e+07,2548000,51550170.0,51550170.0,89500000.0,Aprobado
1,2026-01-06 11:58:15.470,2026-01-06 11:58:08.607,2026-01-06 11:58:07.857,6221346,Vehiculo,Y,Vehículo + Tarjeta (Cruzada),"2926439,82",0,2.878003e+07,0.000000e+00,"1354833,25",60289170.0,60289170.0,28700000.0,Denegado
2,2026-01-06 14:31:49.873,2026-01-06 14:31:43.453,2026-01-06 14:31:42.687,6255146,Vehiculo,A,Vehículo + Tarjeta (Cruzada),10000000,41450000,1.561350e+08,1.500000e+07,"141242062,5",207539170.0,207539170.0,156100000.0,Aprobado
3,2026-01-16 09:48:38.793,2026-01-07 08:22:12.023,2026-01-07 08:22:11.083,6265901,Vehiculo,B,Vehículo + Tarjeta (Cruzada),"4049117,1","7800000,075",1.144000e+08,7.800000e+06,2999346,84289170.0,84289170.0,114400000.0,Aprobado
4,2026-01-16 17:03:08.820,2026-01-07 16:55:20.250,2026-01-07 16:55:19.497,6280624,Vehiculo,C,Vehículo + Tarjeta (Cruzada),3780000,6963750,8.000000e+07,6.963750e+06,2450000,80000000.0,80289170.0,80000000.0,Aprobado
5,2026-01-06 11:44:51.663,2026-01-06 11:44:44.277,2026-01-06 11:44:43.527,6285913,Vehiculo,B,Vehículo + Tarjeta (Cruzada),"6841011,06",5999500,1.319890e+08,0.000000e+00,900000,20000000.0,20289170.0,30200000.0,Denegado
6,2026-01-06 11:23:12.117,2026-01-06 11:23:06.313,2026-01-06 11:23:05.577,6309679,Vehiculo,S,Vehículo + Tarjeta (Cruzada),10000000,6090000,1.339800e+08,0.000000e+00,8496771,121800000.0,122089170.0,133900000.0,Denegado
7,2026-01-07 14:20:02.383,2026-01-07 14:19:56.993,2026-01-07 14:19:56.240,6310831,Vehiculo,Y,Vehículo + Tarjeta (Cruzada),3240000,2000000,3.850000e+07,0.000000e+00,2400000,35000000.0,35289170.0,38500000.0,Denegado
8,2026-01-08 15:51:20.417,2026-01-08 15:51:14.793,2026-01-08 15:51:14.027,6235171,Vehiculo,C,Vehículo + Tarjeta (Cruzada),"5015613,96","6599250,075",8.000000e+07,6.599250e+06,"3250860,9",85289170.0,85289170.0,80000000.0,Aprobado
9,2026-01-19 18:02:04.650,2026-01-07 15:46:03.730,2026-01-07 15:46:02.997,6246599,Vehiculo,A,Vehículo + Tarjeta (Cruzada),10000000,"12490000,1",1.373900e+08,1.249000e+07,"6808281,2",60289170.0,60289170.0,137300000.0,Aprobado


### 5. Bloque csv

In [33]:
print("5. Exportando resultados cruzados a Excel y CSV...")

# Definimos las rutas de salida
ruta_excel = r'\\192.168.60.149\gerencia de riesgo\DIRECCION DE MODELOS DE RIESGO DE CREDITO\Oscar Camargo\cruce_motor_cartera.xlsx'
ruta_csv = r'\\192.168.60.149\gerencia de riesgo\DIRECCION DE MODELOS DE RIESGO DE CREDITO\Oscar Camargo\cruce_motor_cartera.csv'

# =====================================================================
# ARCHIVO 1: Guardar como EXCEL
# =====================================================================
try:
    print("Generando archivo Excel (esto puede tomar unos segundos)...")
    with pd.ExcelWriter(ruta_excel, engine='openpyxl') as writer:
        df_final.to_excel(writer, index=False, sheet_name='Data_Cruzada')
        
        # Auto-ajuste de columnas
        worksheet = writer.sheets['Data_Cruzada']
        for column in worksheet.columns:
            max_length = 0
            column_letter = column[0].column_letter
            for cell in column:
                try:
                    if len(str(cell.value)) > max_length:
                        max_length = len(str(cell.value))
                except:
                    pass
            worksheet.column_dimensions[column_letter].width = (max_length + 2)
            
    print(f"✅ ¡Excel guardado con éxito!: {ruta_excel}")
except Exception as e:
    print(f"Error al guardar el Excel: {e}")

# =====================================================================
# ARCHIVO 2: Guardar como CSV 
# =====================================================================
print("\nGenerando archivo CSV...")
df_final.to_csv(ruta_csv, index=False, sep=';', encoding='utf-8-sig')
print(f"✅ ¡CSV guardado con éxito!: {ruta_csv}")

5. Exportando resultados cruzados a Excel y CSV...
Generando archivo Excel (esto puede tomar unos segundos)...
✅ ¡Excel guardado con éxito!: \\192.168.60.149\gerencia de riesgo\DIRECCION DE MODELOS DE RIESGO DE CREDITO\Oscar Camargo\cruce_motor_cartera.xlsx

Generando archivo CSV...
✅ ¡CSV guardado con éxito!: \\192.168.60.149\gerencia de riesgo\DIRECCION DE MODELOS DE RIESGO DE CREDITO\Oscar Camargo\cruce_motor_cartera.csv


### 6. Parquet

In [34]:
print("6. Exportando resultados cruzados a formato Parquet...")

# =====================================================================
# ARCHIVO 3: Guardar como PARQUET
# =====================================================================
ruta_parquet = r'\\192.168.60.149\gerencia de riesgo\DIRECCION DE MODELOS DE RIESGO DE CREDITO\Oscar Camargo\sc_cp.parquet'

df_final.to_parquet(ruta_parquet, index=False)

print(f"✅ ¡Parquet guardado con éxito en la red!: {ruta_parquet}")

6. Exportando resultados cruzados a formato Parquet...
✅ ¡Parquet guardado con éxito en la red!: \\192.168.60.149\gerencia de riesgo\DIRECCION DE MODELOS DE RIESGO DE CREDITO\Oscar Camargo\sc_cp.parquet


In [35]:
# Muestra qué productos del motor coinciden con el Producto Principal de Bizagi
df_final[df_final['MontoSolicitado'].notnull()]['Producto'].value_counts()

Producto
Vehiculo                    259684
Libranza                        13
TarjetaCreditoSecundario        12
TarjetaCredito                   3
Name: count, dtype: int64